In [ ]:
!pip install -q torch transformers accelerate peft datasets

In [ ]:
!pip install -q -U torchao peft

In [ ]:
from datasets import Dataset
import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

model_adi = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_adi)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

# 1. Küçük örnek veri seti (Soru -> Net teknik cevap)
ornekler = [
    {
        "soru": "Sinyal kaybı 15 saniyeyi aşarsa ne olur?",
        "cevap": "Otomatik Eve Donus (RTH) proseduru baslatilir.",
    },
    {
        "soru": "Batarya voltaji 22.4V altina duserse ne yapilir?",
        "cevap": (
            "Kritik uyari verilir, faydasiz yuk enerjisi kesilir ve guc ucus"
            " kontrolune aktarilir."
        ),
    },
    {
        "soru": "Egzoz sicakligi 120 dereceyi gecerse ne olur?",
        "cevap": "Otomatik guc kisma (derating) modu devreye girer.",
    },
    {
        "soru": "Karbon-seramik fren soguma suresi nedir?",
        "cevap": "Inis sonrasi minimum 20 dakikadir.",
    },
    {
        "soru": "Buz cozucu sistem ne zaman aktiflesir?",
        "cevap": (
            "Dis sicaklik +2 derecenin altina inip nem %80'i gecince otonom"
            " aktiflesir."
        ),
    },
    {
        "soru": "Hidrolik inis takimlari hangi irtifada acilir?",
        "cevap": "Yerden 150 metre irtifada acilir.",
    },
]


def format_ve_tokenize(ornek):
  metin = (
      f"Soru: {ornek['soru']}\nCevap: {ornek['cevap']}{tokenizer.eos_token}"
  )
  tokens = tokenizer(
      metin, truncation=True, max_length=128, padding="max_length"
  )
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens


ham_veri = Dataset.from_list(ornekler)
egitim_verisi = ham_veri.map(
    format_ve_tokenize, remove_columns=["soru", "cevap"]
)

# 2. Temel modeli yukle ve ana agirliklari dondur
base_model = AutoModelForCausalLM.from_pretrained(
    model_adi, torch_dtype=torch.float16, device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # LoRA matrislerinin rank degeri (kucuk rank = daha az parametre)
    lora_alpha=16,  # Adaptor olcekleme katsayisi
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # Attention katmanlarindaki projeksiyonlara ekle
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

# 3. Kisa bir egitim dongusu
training_args = TrainingArguments(
    output_dir="./lora_iha_cikti",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=egitim_verisi,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

# 4. Egitim sonrasi test
test_prompt = "Soru: Sinyal kaybi 15 saniyeyi asarsa ne olur?\nCevap:"
girdiler = tokenizer(test_prompt, return_tensors="pt").to(lora_model.device)
with torch.no_grad():
  cikti = lora_model.generate(
      **girdiler, max_new_tokens=40, temperature=0.1, do_sample=True
  )

print(
    "\nMODEL CIKTISI:\n",
    tokenizer.decode(cikti[0], skip_special_tokens=True),
)

W1122 09:18:54.175000 1438 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 09:18:54.269000 1438 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Step,Training Loss
5,4.356949
10,4.133690
15,4.234779



MODEL CIKTISI:
 Soru: Sinyal kaybi 15 saniyeyi asarsa ne olur?
Cevap: Sıfır

Bu soru, bir bilgisayarın sinyal kaybına dayanıklı hızını ölçmek için kullanılan bir test olarak kabul edilir. Bu test


In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

model_adi = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_adi)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 1. 18 Satırlık KIZILELMA-X Teknik Operasyon Veri Seti
ornekler = [
    {"soru": "Sinyal kaybı 15 saniyeyi aşarsa ne olur?", "cevap": "Otomatik Eve Dönüş (RTH) prosedürü başlatılır."},
    {"soru": "Batarya voltajı 22.4V altına düşerse ne yapılır?", "cevap": "Kritik düşük voltaj uyarısı verilir, faydalı yük enerjisi kesilerek yalnızca uçuş kontrol bilgisayarına güç aktarılır."},
    {"soru": "Egzoz sıcaklığı 120 dereceyi geçerse ne olur?", "cevap": "Otomatik güç kısma (derating) modu devreye girer."},
    {"soru": "Ana itki sistemi maksimum kaç derece egzoz sıcaklığında sürekli çalışabilir?", "cevap": "Maksimum 115 derece egzoz gazı sıcaklığında sürekli çalışabilir."},
    {"soru": "Karbon-seramik fren soğuma süresi nedir?", "cevap": "İniş sonrası minimum 20 dakikadır; bu süre dolmadan yeni kalkış izni verilmez."},
    {"soru": "Buz çözücü sistem ne zaman aktifleşir?", "cevap": "Dış hava sıcaklığı +2 derecenin altına düştüğünde ve nem oranı %80'i geçtiğinde otonom olarak aktifleşir."},
    {"soru": "Hidrolik iniş takımları hangi irtifada açılır?", "cevap": "Yerden 150 metre irtifada açılır."},
    {"soru": "Yer kontrol istasyonu ile görüş hattı haberleşmesi hangi bant üzerinden sağlanır?", "cevap": "C-Bant üzerinden sağlanır."},
    {"soru": "Otopilot modunda maksimum uçuş irtifası nedir?", "cevap": "Maksimum irtifa 15.000 feet olarak sınırlandırılmıştır."},
    {"soru": "Rüzgar hızı kaç knot üzerine çıkarsa sistem RTL moduna geçer?", "cevap": "Rüzgar hızı 35 knot üzerine çıkarsa otomatik olarak eve dönüş (RTL) moduna geçer."},
    {"soru": "Görüş hattı (LOS) datalink menzili ne kadardır?", "cevap": "150 km menzile sahiptir."},
    {"soru": "Lityum-Polimer batarya paketleri tam şarjda kaç saat uçuş sağlar?", "cevap": "Tam şarjda 4.5 saat uçuş süresi sağlar."},
    {"soru": "Kritik voltaj uyarısında hangi sistemlerin gücü kesilir?", "cevap": "Kamera ve radar gibi faydalı yüklerin gücü kesilir."},
    {"soru": "Kanat hücum kenarlarında hangi tip buz önleme sistemi bulunur?", "cevap": "Elektro-termal buz çözücü sistem bulunur."},
    {"soru": "İniş sonrası 20 dakika geçmeden kalkış izni verilir mi?", "cevap": "Hayır, karbon-seramik frenlerin soğuması için minimum 20 dakika beklenmeden yeni kalkış izni verilmez."},
    {"soru": "KIZILELMA-X platformunda sinyal kaybı limiti kaç saniyedir?", "cevap": "15 saniyedir; bu süre aşılırsa RTH devreye girer."},
    {"soru": "Dış hava sıcaklığı +1 derece ve nem %85 ise buz çözücü çalışır mı?", "cevap": "Evet, sıcaklık +2 derecenin altında ve nem %80'in üzerinde olduğu için otonom olarak aktifleşir."},
    {"soru": "Faydalı yük gücü hangi voltaj eşiğinde kesilir?", "cevap": "Ana aviyonik batarya voltajı 22.4V altına düştüğünde kesilir."}
]

# 2. Modelin Kendi Chat Template'i ile Formatlama
def format_ve_tokenize(ornek):
    mesajlar = [
        {"role": "system", "content": "Sen KIZILELMA-X İHA sistem mühendisliği asistanısın. Teknik sorulara net ve kısa yanıt ver."},
        {"role": "user", "content": ornek["soru"]},
        {"role": "assistant", "content": ornek["cevap"]}
    ]
    metin = tokenizer.apply_chat_template(mesajlar, tokenize=False)
    tokens = tokenizer(metin, truncation=True, max_length=160, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

ham_veri = Dataset.from_list(ornekler)
egitim_verisi = ham_veri.map(format_ve_tokenize, remove_columns=["soru", "cevap"])

# 3. Temel Modeli Yükle
base_model = AutoModelForCausalLM.from_pretrained(
    model_adi, torch_dtype=torch.float16, device_map="auto"
)

def soru_sor(model, soru):
    mesajlar = [
        {"role": "system", "content": "Sen KIZILELMA-X İHA sistem mühendisliği asistanısın. Teknik sorulara net ve kısa yanıt ver."},
        {"role": "user", "content": soru}
    ]
    prompt = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    girdiler = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        cikti = model.generate(**girdiler, max_new_tokens=60, do_sample=False)
    yeni_tokenler = cikti[0][girdiler.input_ids.shape[1]:]
    return tokenizer.decode(yeni_tokenler, skip_special_tokens=True)

# --- EĞİTİM ÖNCESİ (BASE) TEST ---
test_sorulari = [
    "Sinyal kaybı 15 saniyeyi aşarsa ne olur?",
    "Batarya voltajı 22.4V altına düşerse ne yapılır?",
    "Karbon-seramik fren soğuma süresi nedir?"
]

print("=== EĞİTİM ÖNCESİ (BASE MODEL) CEVAPLARI ===")
base_cevaplar = {}
for s in test_sorulari:
    base_cevaplar[s] = soru_sor(base_model, s)
    print(f"Soru : {s}\nCevap: {base_cevaplar[s]}\n")

# 4. LoRA Adaptörünü Ekle ve Eğit
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                # Daha iyi öğrenme için rank 16
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]  # Tüm attention projeksiyonları
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./lora_iha_cikti",
    per_device_train_batch_size=2,
    num_train_epochs=15,         # Küçük veri setini ağırlıklara kazımak için 15 epoch
    learning_rate=3e-4,
    logging_steps=15,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=egitim_verisi,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

# --- EĞİTİM SONRASI (LoRA) KARŞILAŞTIRMALI TEST ---
print("\n" + "="*60)
print("=== EĞİTİM ÖNCESİ vs. SONRASI KARŞILAŞTIRMA ===")
print("="*60)
for s in test_sorulari:
    lora_cevap = soru_sor(lora_model, s)
    print(f"SORU        : {s}")
    print(f"[BASE]      : {base_cevaplar[s]}")
    print(f"[LoRA-EĞİT] : {lora_cevap}")
    print("-" * 60)

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== EĞİTİM ÖNCESİ (BASE MODEL) CEVAPLARI ===
Soru : Sinyal kaybı 15 saniyeyi aşarsa ne olur?
Cevap: Sinyal kaybı, sinyallerin bir anda hızla azaldığı durumdır. Bu durumda, sinyal kaybı 15 saniyeye ulaşana kadar sinyallerin hızla azalmayacağını belirtir. Ancak,

Soru : Batarya voltajı 22.4V altına düşerse ne yapılır?
Cevap: Batarya voltajı 22.4V altına düşerken, bu durumda aşağıdaki adımları takip edebilirsiniz:

1. Batarya kontrolü: İlk olarak, bataryayı kontrol etmek için bir elektrik analitörü kullan

Soru : Karbon-seramik fren soğuma süresi nedir?
Cevap: Carbon-seramik frenlerin soğuma süresini belirlemek için birkaç faktör dahil edilmelidir:

1. Fren tipi: Dışarıdan gelen kuvveti koruyabildiği, içten bir fren daha güvenli.



trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


Step,Training Loss
15,2.694234
30,1.051690
45,0.644721
60,0.338417
75,0.164152
90,0.101555
105,0.068556
120,0.054148
135,0.046371



=== EĞİTİM ÖNCESİ vs. SONRASI KARŞILAŞTIRMA ===
SORU        : Sinyal kaybı 15 saniyeyi aşarsa ne olur?
[BASE]      : Sinyal kaybı, sinyallerin bir anda hızla azaldığı durumdır. Bu durumda, sinyal kaybı 15 saniyeye ulaşana kadar sinyallerin hızla azalmayacağını belirtir. Ancak,
[LoRA-EĞİT] : Otomatik Eve Dönüş (RTH) prosedürü başlatılır.
------------------------------------------------------------
SORU        : Batarya voltajı 22.4V altına düşerse ne yapılır?
[BASE]      : Batarya voltajı 22.4V altına düşerken, bu durumda aşağıdaki adımları takip edebilirsiniz:

1. Batarya kontrolü: İlk olarak, bataryayı kontrol etmek için bir elektrik analitörü kullan
[LoRA-EĞİT] : Kritik düşük voltaj uyarısı verilir, faydalı yük enerjisi kesilerek yalnızca uçuş kontrol bilgisayarına güç aktarılır.
------------------------------------------------------------
SORU        : Karbon-seramik fren soğuma süresi nedir?
[BASE]      : Carbon-seramik frenlerin soğuma süresini belirlemek için birkaç faktör dahil